# eICU Demo EDA for FairnessOps

This notebook performs first-pass EDA for the FairnessOps MVP using eICU demo tables.

**Goals**
- Verify table schemas and join keys
- Profile protected attributes and missingness
- Inspect feature coverage for labs/vitals/APACHE
- Produce quick subgroup mortality summaries to guide modeling

In [12]:
from pathlib import Path
import pandas as pd
import numpy as np
import os

pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 140)

# Remote-runtime friendly relative paths
CWD = Path.cwd()
PROJECT_ROOT = CWD.parent if CWD.name == "scripts" else CWD

# Dataset is expected under project root
DATA_DIR = PROJECT_ROOT / "eicu-collaborative-research-database-demo-2.0.1"

# If not found, try common alternate when notebook runs from deeper directory
if not DATA_DIR.exists() and (PROJECT_ROOT.parent / "eicu-collaborative-research-database-demo-2.0.1").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
    DATA_DIR = PROJECT_ROOT / "eicu-collaborative-research-database-demo-2.0.1"

print("cwd:", CWD)
print("project_root:", PROJECT_ROOT)
print("dataset_path:", DATA_DIR)
print("dataset_exists:", DATA_DIR.exists())
if DATA_DIR.exists():
    print("dataset sample files:", sorted([p.name for p in DATA_DIR.iterdir()])[:8])

Project root: c:\Users\venka\Desktop\Fairness Ops
Dataset path: c:\Users\venka\Desktop\Fairness Ops/eicu-collaborative-research-database-demo-2.0.1
Dataset exists: False


In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 140)

# Explicit paths (as provided)
PROJECT_ROOT = Path(r"c:\Users\venka\Desktop\Fairness Ops")
DATA_DIR = PROJECT_ROOT / "eicu-collaborative-research-database-demo-2.0.1"

print("Project root:", PROJECT_ROOT)
print("Dataset path:", DATA_DIR)
print("Dataset exists:", DATA_DIR.exists())

Project root: c:\Users\venka\Desktop\Fairness Ops
Dataset path: c:\Users\venka\Desktop\Fairness Ops/eicu-collaborative-research-database-demo-2.0.1
Dataset exists: False


In [10]:
# Core tables for MVP (relative-path + csv/csv.gz fallback)
def read_table(stem: str) -> pd.DataFrame:
    csv_path = DATA_DIR / f"{stem}.csv"
    gz_path = DATA_DIR / f"{stem}.csv.gz"
    if csv_path.exists():
        return pd.read_csv(csv_path)
    if gz_path.exists():
        return pd.read_csv(gz_path)
    raise FileNotFoundError(f"Missing {stem}.csv/.csv.gz in {DATA_DIR}")

patient = read_table("patient")
hospital = read_table("hospital")
lab = read_table("lab")
vital = read_table("vitalPeriodic")
apache = read_table("apachePatientResult")

print("Loaded tables:")
for name, df in {
    "patient": patient,
    "hospital": hospital,
    "lab": lab,
    "vitalPeriodic": vital,
    "apachePatientResult": apache,
}.items():
    print(f"- {name:20s} shape={df.shape}")

FileNotFoundError: [Errno 2] No such file or directory: 'c:\\Users\\venka\\Desktop\\Fairness Ops/eicu-collaborative-research-database-demo-2.0.1/patient.csv'

In [ ]:
def table_summary(df: pd.DataFrame, name: str, key_cols=None):
    out = {
        "table": name,
        "rows": len(df),
        "cols": df.shape[1],
        "n_missing_cells": int(df.isna().sum().sum()),
    }
    if key_cols:
        for c in key_cols:
            if c in df.columns:
                out[f"unique_{c}"] = int(df[c].nunique(dropna=True))
                out[f"dup_{c}"] = int(df[c].duplicated().sum())
    return out

summaries = pd.DataFrame([
    table_summary(patient, "patient", ["patientunitstayid", "hospitalid"]),
    table_summary(hospital, "hospital", ["hospitalid"]),
    table_summary(lab, "lab", ["patientunitstayid"]),
    table_summary(vital, "vitalPeriodic", ["patientunitstayid"]),
    table_summary(apache, "apachePatientResult", ["patientunitstayid"]),
])

summaries

,table,rows,cols,n_missing_cells,unique_patientunitstayid,dup_patientunitstayid,unique_hospitalid,dup_hospitalid
0,patient,2520,29,2579,2520.0,0.0,186.0,2334.0
1,hospital,186,4,46,NaN,NaN,186.0,0.0
2,lab,434660,10,40545,2444.0,432216.0,NaN,NaN
3,vitalPeriodic,1634960,19,18662595,2375.0,1632585.0,NaN,NaN
4,apachePatientResult,3676,23,7940,1838.0,1838.0,NaN,NaN


In [ ]:
# Join key overlap checks
p_ids = set(patient["patientunitstayid"].dropna().astype(int).unique())

coverage = {
    "lab_overlap_pct": 100 * (lab["patientunitstayid"].isin(p_ids).mean()),
    "vital_overlap_pct": 100 * (vital["patientunitstayid"].isin(p_ids).mean()),
    "apache_overlap_pct": 100 * (apache["patientunitstayid"].isin(p_ids).mean()),
    "patient_hospital_overlap_pct": 100 * (patient["hospitalid"].isin(set(hospital["hospitalid"])).mean()),
}

pd.DataFrame([coverage]).round(2)

,lab_overlap_pct,vital_overlap_pct,apache_overlap_pct,patient_hospital_overlap_pct
0,100.0,100.0,100.0,100.0


In [ ]:
# Target + protected attribute profiling
p = patient.copy()
p["y_true"] = (p["hospitaldischargestatus"].astype(str).str.lower() == "expired").astype(int)

# Age cleanup (eICU can contain strings like '> 89')
p["age_num"] = pd.to_numeric(
    p["age"].astype(str).str.replace(">", "", regex=False).str.strip(),
    errors="coerce",
)
p["age_group"] = pd.cut(
    p["age_num"],
    bins=[0, 45, 65, 200],
    labels=["under_45", "45_65", "over_65"],
    right=True,
)

# Region from hospital
p = p.merge(hospital[["hospitalid", "region", "teachingstatus"]], on="hospitalid", how="left")

protected_cols = ["ethnicity", "gender", "age_group", "region"]

print(f"Mortality prevalence (y_true=1): {p['y_true'].mean():.3%}")
print("\nMissingness for protected columns:")
print((p[protected_cols].isna().mean() * 100).round(2).rename("missing_%"))

print("\nSample subgroup counts:")
for c in protected_cols:
    print(f"\n{c}:")
    # Works for categorical columns (e.g., age_group) and object columns
    grp = p[c].astype("object").fillna("Unknown")
    print(grp.value_counts(dropna=False).head(10))

Mortality prevalence (y_true=1): 8.413%

Missingness for protected columns:
ethnicity    1.55
gender       0.16
age_group    0.16
region       8.33
Name: missing_%, dtype: float64

Sample subgroup counts:

ethnicity:
ethnicity
Caucasian           2010
African American     231
Hispanic             115
Other/Unknown         83
Unknown               39
Asian                 30
Native American       12
Name: count, dtype: int64

gender:
gender
Male       1508
Female     1008
Unknown       4
Name: count, dtype: int64

age_group:
age_group
over_65     1295
45_65        787
under_45     434
Unknown        4
Name: count, dtype: int64

region:
region
Midwest      807
South        738
West         606
Unknown      210
Northeast    159
Name: count, dtype: int64


In [ ]:
# Candidate feature coverage (labs + vitals + apachescore)
key_labs = ["WBC x1000", "creatinine", "glucose", "sodium", "potassium", "lactate", "pH"]

lab_subset = lab[lab["labname"].isin(key_labs)].copy()
lab_agg = (
    lab_subset.groupby(["patientunitstayid", "labname"]) ["labresult"]
    .median()
    .unstack()
)
lab_agg.columns = [f"lab_{c}" for c in lab_agg.columns]

vital_cols = ["temperature", "sao2", "heartrate", "respiration", "systemicsystolic", "systemicdiastolic", "systemicmean"]
vital_agg = vital.groupby("patientunitstayid")[vital_cols].median()
vital_agg.columns = [f"vital_{c}" for c in vital_agg.columns]

apache_agg = apache[["patientunitstayid", "apachescore"]].drop_duplicates("patientunitstayid").set_index("patientunitstayid")

features = lab_agg.join(vital_agg, how="outer").join(apache_agg, how="outer")
coverage = (features.notna().mean() * 100).sort_values(ascending=False).rename("non_null_%").round(2)

print("Feature matrix shape:", features.shape)
coverage

Feature matrix shape: (2471, 14)


lab_creatinine             96.03
lab_sodium                 95.87
vital_heartrate            95.67
lab_glucose                95.22
lab_potassium              94.74
vital_sao2                 94.29
vital_respiration          87.49
apachescore                74.38
lab_pH                     40.19
lab_lactate                33.95
vital_systemicmean         17.52
vital_systemicsystolic     17.40
vital_systemicdiastolic    17.36
vital_temperature           6.88
Name: non_null_%, dtype: float64

In [ ]:
# Quick descriptive mortality by subgroup (not final fairness metric)
def mortality_by_group(df, group_col, min_n=20):
    group_series = df[group_col].astype("object").fillna("Unknown")
    tmp = (
        df.assign(group=group_series)
          .groupby("group")
          .agg(n=("y_true", "size"), mortality_rate=("y_true", "mean"))
          .sort_values("n", ascending=False)
    )
    tmp["mortality_rate"] = tmp["mortality_rate"].round(4)
    return tmp[tmp["n"] >= min_n]

for c in ["ethnicity", "gender", "age_group", "region"]:
    print(f"\n=== {c} ===")
    display(mortality_by_group(p, c, min_n=20).head(15))


=== ethnicity ===


,n,mortality_rate
group,,
Caucasian,2010,0.0861
African American,231,0.0390
Hispanic,115,0.0957
Other/Unknown,83,0.1325
Unknown,39,0.1026
Asian,30,0.1333



=== gender ===


,n,mortality_rate
group,,
Male,1508,0.0822
Female,1008,0.0863



=== age_group ===


,n,mortality_rate
group,,
over_65,1295,0.1174
45_65,787,0.0661
under_45,434,0.0161



=== region ===


,n,mortality_rate
group,,
Midwest,807,0.0719
South,738,0.0854
West,606,0.0825
Unknown,210,0.1048
Northeast,159,0.1195


In [ ]:
# Optional: save EDA artifacts for reuse
out_dir = PROJECT_ROOT / "runs"
out_dir.mkdir(parents=True, exist_ok=True)

summaries.to_csv(out_dir / "eda_table_summaries.csv", index=False)
coverage.reset_index().rename(columns={"index": "feature"}).to_csv(out_dir / "eda_feature_coverage.csv", index=False)

print("Saved:")
print("-", out_dir / "eda_table_summaries.csv")
print("-", out_dir / "eda_feature_coverage.csv")

Saved:
- c:\Users\venka\Desktop\Fairness Ops\runs\eda_table_summaries.csv
- c:\Users\venka\Desktop\Fairness Ops\runs\eda_feature_coverage.csv


## Modeling Spec (MVP): Label and Features

### Prediction target (label)
- **`y_true`**: in-hospital mortality
  - Derived from `patient.hospitaldischargestatus`
  - Mapping: `Expired -> 1`, all other outcomes -> `0`

### Modeling grain
- One row per **`patientunitstayid`** (ICU stay)

### Features we will use in the baseline Logistic Regression

#### From `lab.csv` (aggregated to patient level, median per stay)
- **`lab_creatinine`**: kidney function marker; elevated values often indicate renal dysfunction and higher severity.
- **`lab_sodium`**: electrolyte balance; abnormal sodium is associated with worse outcomes.
- **`lab_glucose`**: blood sugar; extreme values can reflect physiologic stress and poor prognosis.
- **`lab_potassium`**: electrolyte critical for cardiac function; abnormal levels are clinically high risk.

#### From `vitalPeriodic.csv` (aggregated to patient level, median per stay)
- **`vital_heartrate`**: sustained tachycardia/bradycardia can signal instability.
- **`vital_sao2`**: oxygen saturation; lower values indicate respiratory compromise.
- **`vital_respiration`**: respiratory rate; elevated/depressed rates can indicate acute deterioration.

#### From `apachePatientResult.csv` (deduplicated to one row per stay)
- **`apachescore`**: overall ICU severity score summarizing acute physiologic illness burden.

### Protected attributes used for fairness auditing (not model inputs)
- **`ethnicity`**
- **`gender`**
- **`age_group`** (derived from `age`)
- **`region`** (from `hospital.csv`)
- **`insurance`**, **`language`** set to `Unknown` in demo if unavailable

### Notes
- Missing numeric values in modeling features will be median-imputed.
- We intentionally do **not** train on protected attributes in the baseline model.
- Fairness metrics are computed after prediction by slicing performance across protected groups.

In [ ]:
na_cols = patient.isna().sum()
na_cols = na_cols[na_cols > 0].sort_values(ascending=False)
na_report = pd.DataFrame({
  "na_count": na_cols,
  "na_pct": (na_cols / len(patient) * 100).round(2)
})
print(na_report)

                           na_count  na_pct
dischargeweight                1284   50.95
hospitaladmitsource             594   23.57
apacheadmissiondx               299   11.87
admissionweight                 198    7.86
admissionheight                  69    2.74
ethnicity                        39    1.55
hospitaldischargelocation        32    1.27
hospitaldischargestatus          28    1.11
unitadmitsource                  22    0.87
age                               4    0.16
gender                            4    0.16
unitdischargelocation             4    0.16
unitdischargestatus               2    0.08


## Build Canonical Dataset (One Row Per ICU Stay)

This section creates the modeling-ready canonical table keyed by `patientunitstayid`.

- Anchor: `patient`
- Joins: `hospital`, aggregated `lab`, aggregated `vitalPeriodic`, deduplicated `apachePatientResult`
- Target: `y_true` (in-hospital mortality)
- Protected attributes retained for fairness auditing
- Saves outputs to `runs/canonical_dataset.csv` and `runs/canonical_dataset.parquet`

In [ ]:
# 1) Start from patient anchor and derive target/protected attrs
canonical = patient.copy()

canonical["y_true"] = (
    canonical["hospitaldischargestatus"].astype(str).str.lower() == "expired"
).astype(int)

canonical["age_num"] = pd.to_numeric(
    canonical["age"].astype(str).str.replace(">", "", regex=False).str.strip(),
    errors="coerce",
)
canonical["age_group"] = pd.cut(
    canonical["age_num"],
    bins=[0, 45, 65, 200],
    labels=["under_45", "45_65", "over_65"],
    right=True,
).astype("object").fillna("Unknown")

canonical["ethnicity"] = canonical["ethnicity"].astype("object").fillna("Unknown")
canonical["gender"] = canonical["gender"].astype("object").fillna("Unknown")

# Demo placeholders when unavailable in this subset
canonical["insurance"] = "Unknown"
canonical["language"] = "Unknown"

# bring in region and teaching status from hospital
canonical = canonical.merge(
    hospital[["hospitalid", "region", "teachingstatus"]],
    on="hospitalid",
    how="left",
)
canonical["region"] = canonical["region"].astype("object").fillna("Unknown")

print("Anchor shape:", canonical.shape)
print("Unique patientunitstayid:", canonical["patientunitstayid"].nunique())

In [ ]:
# 2) Aggregate lab features to one row per patient stay
selected_labs = ["creatinine", "sodium", "glucose", "potassium", "lactate", "pH", "WBC x1000"]

lab_agg = (
    lab[lab["labname"].isin(selected_labs)]
    .groupby(["patientunitstayid", "labname"]) ["labresult"]
    .median()
    .unstack()
)
lab_agg.columns = [f"lab_{c}" for c in lab_agg.columns]

print("Lab aggregated shape:", lab_agg.shape)
lab_agg.head()

In [ ]:
# 3) Aggregate vital features to one row per patient stay
selected_vitals = [
    "heartrate", "sao2", "respiration",
    "systemicsystolic", "systemicdiastolic", "systemicmean", "temperature"
]

vital_agg = vital.groupby("patientunitstayid")[selected_vitals].median()
vital_agg.columns = [f"vital_{c}" for c in vital_agg.columns]

print("Vital aggregated shape:", vital_agg.shape)
vital_agg.head()

In [ ]:
# 4) Deduplicate APACHE to one row per stay (keep first non-null by sort order)
apache_slim = (
    apache[["patientunitstayid", "apachescore"]]
    .sort_values(["patientunitstayid", "apachescore"], na_position="last")
    .drop_duplicates("patientunitstayid", keep="first")
    .set_index("patientunitstayid")
)

print("APACHE slim shape:", apache_slim.shape)
apache_slim.head()

In [ ]:
# 5) Join everything into canonical table
canonical = (
    canonical
    .merge(lab_agg, on="patientunitstayid", how="left")
    .merge(vital_agg, on="patientunitstayid", how="left")
    .merge(apache_slim, on="patientunitstayid", how="left")
)

# Keep a focused set of columns for modeling + fairness
model_feature_cols = [
    "lab_creatinine", "lab_sodium", "lab_glucose", "lab_potassium",
    "vital_heartrate", "vital_sao2", "vital_respiration", "apachescore"
]

audit_cols = [
    "patientunitstayid", "hospitalid", "y_true",
    "ethnicity", "gender", "age_group", "insurance", "language", "region",
]

canonical_final = canonical[audit_cols + model_feature_cols].copy()

print("Canonical final shape:", canonical_final.shape)
print("Rows preserved from patient:", canonical_final.shape[0] == patient.shape[0])
print("Unique patientunitstayid:", canonical_final["patientunitstayid"].nunique())
print("Duplicate patientunitstayid:", canonical_final["patientunitstayid"].duplicated().sum())

canonical_final.head()

In [ ]:
# 6) Missingness report + save outputs
canonical_missing = (
    canonical_final.isna().mean().mul(100).round(2).rename("missing_pct").sort_values(ascending=False)
)

display(canonical_missing)

out_dir = PROJECT_ROOT / "runs"
out_dir.mkdir(parents=True, exist_ok=True)

csv_path = out_dir / "canonical_dataset.csv"
parquet_path = out_dir / "canonical_dataset.parquet"

canonical_final.to_csv(csv_path, index=False)
print("Saved CSV:", csv_path)

# Parquet is optional (requires pyarrow or fastparquet)
try:
    canonical_final.to_parquet(parquet_path, index=False)
    print("Saved Parquet:", parquet_path)
except Exception as e:
    print("Parquet skipped:", e)
    print("Tip: pip install pyarrow")